In [8]:
import pandas as pd

# Load the dataset
df = pd.read_csv("dados_manutencao.csv")

# Display the first 5 rows
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

# Print the column names and their data types
print(df.info())

| Data de Produção Acumulada   | Cod. Ordem   | Cod Recurso   | Cod Produto   | Qt. Total Acumulada Produzida até a data específica   | Qt. Acumulada Refugada até a data específica   | Qtd. Acumulada total Retrabalhada até a data específica   | Fator Un.   | Cód. Un.   | Descrição da massa (Composto)   | Consumo total de Massa Acumulada   | Tempo Restante para Manutenção   |
|:-----------------------------|:-------------|:--------------|:--------------|:------------------------------------------------------|:-----------------------------------------------|:----------------------------------------------------------|:------------|:-----------|:--------------------------------|:-----------------------------------|:---------------------------------|
| 2023-09-08                   | 4458603      | IJ-117        | SA02961       | 87                                                    | 98                                             | 14                                                        |

In [9]:
    """
         Vamos Remover as colunas Data de Produção Acumulada, Cod. Ordem, Cod Recurso, Fator Un., Cód. Un., e Descrição da massa (Composto), pois elas não são relevantes para a previsão do Tempo Restante para Manutenção. As colunas Qt. Total Acumulada Produzida até a data específica, Qt. Acumulada Refugada até a data específica, e Qtd. Acumulada total Retrabalhada até a data específica representam quantidades acumuladas até uma data específica, então vou renomeá-las para Qtd_Produzida, Qtd_Refugada e Qtd_Retrabalhada, respectivamente.
    """
    # Rename columns
df = df.rename(columns={
    'Qt. Total Acumulada Produzida até a data específica': 'Qtd_Produzida',
    'Qt. Acumulada Refugada até a data específica': 'Qtd_Refugada',
    'Qtd. Acumulada total Retrabalhada até a data específica': 'Qtd_Retrabalhada'
})

# Drop unnecessary columns
df = df.drop([
    'Data de Produção Acumulada',
    'Cod. Ordem',
    'Cod Recurso',
    'Fator Un.',
    'Cód. Un.',
    'Descrição da massa (Composto)'
], axis=1)

# Display the first 5 rows
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

# Print the column names and their data types
print(df.info())
    

| Cod Produto   | Qtd_Produzida   | Qtd_Refugada   | Qtd_Retrabalhada   | Consumo total de Massa Acumulada   | Tempo Restante para Manutenção   |
|:--------------|:----------------|:---------------|:-------------------|:-----------------------------------|:---------------------------------|
| SA02961       | 87              | 98             | 14                 | 0.909355                           | -157                             |
| SA05780       | 819             | 1              | 8                  | 0.796544                           | -195                             |
| SA05780       | 84              | 74             | 4                  | 0.686085                           | -153                             |
| SA05780       | 868             | 19             | 47                 | 1.22212                            | -329                             |
| SA02961       | 95              | 69             | 25                 | 0.753044                           | -184         

In [10]:
# Vamos usar as colunas Qtd_Produzida, Qtd_Refugada, Qtd_Retrabalhada, Consumo total de Massa Acumulada, e Cod Produto para prever a coluna Tempo Restante para Manutenção usando o algoritmo Decision Tree Regressor. Vou avaliar o desempenho do modelo usando as métricas Mean Squared Error (MSE) e R-squared.

In [11]:
# One-hot encode categorical variables
df = pd.get_dummies(df, columns=["Cod Produto"])

# Display the first 5 rows
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

# Print the column names and their data types
print(df.info())

| Qtd_Produzida   | Qtd_Refugada   | Qtd_Retrabalhada   | Consumo total de Massa Acumulada   | Tempo Restante para Manutenção   | Cod Produto_SA02004   | Cod Produto_SA02961   | Cod Produto_SA05780   |
|:----------------|:---------------|:-------------------|:-----------------------------------|:---------------------------------|:----------------------|:----------------------|:----------------------|
| 87              | 98             | 14                 | 0.909355                           | -157                             | False                 | True                  | False                 |
| 819             | 1              | 8                  | 0.796544                           | -195                             | False                 | False                 | True                  |
| 84              | 74             | 4                  | 0.686085                           | -153                             | False                 | False                 | True          

In [12]:
    """_summary_
    Vou agora dividir os dados em conjuntos de treinamento e teste e, em seguida, construir e avaliar um modelo de árvore de decisão para prever o Tempo Restante para Manutenção.
    """
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Split into features and target
X = df.drop("Tempo Restante para Manutenção", axis=1)
y = df["Tempo Restante para Manutenção"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
# Train the model
model = DecisionTreeRegressor()
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared: {r2:.2f}")

Mean Squared Error: 19171.49
R-squared: -0.75
